In [ ]:
from collections import deque

print("Program Started")


Program Started


In [ ]:
cfg = {
    "Start": ["Check"],
    "Check": ["Body", "Exit"],
    "Body": ["Check"],
    "Exit": []
}

for node, edges in cfg.items():
    print(node, "->", edges)

Start -> ['Check']
Check -> ['Body', 'Exit']
Body -> ['Check']
Exit -> []


In [ ]:
def reachable_nodes(graph, start):
    visited = set()
    stack = [start]

    while stack:
        node = stack.pop()

        if node not in visited:
            visited.add(node)
            stack.extend(graph.get(node, []))

    return visited


reachable = reachable_nodes(cfg, "Start")

print("Reachable Nodes:", reachable)

Reachable Nodes: {'Exit', 'Start', 'Body', 'Check'}


In [ ]:
def strongly_connected_components(graph):
    index = 0
    indices = {}
    lowlink = {}
    stack = []
    on_stack = set()
    result = []

    nodes = set(graph)

    for neighbors in graph.values():
        nodes.update(neighbors)

    def visit(v):
        nonlocal index

        indices[v] = index
        lowlink[v] = index
        index += 1

        stack.append(v)
        on_stack.add(v)

        for w in graph.get(v, []):

            if w not in indices:
                visit(w)
                lowlink[v] = min(lowlink[v], lowlink[w])

            elif w in on_stack:
                lowlink[v] = min(lowlink[v], indices[w])

        if lowlink[v] == indices[v]:
            component = []

            while True:
                w = stack.pop()
                on_stack.remove(w)
                component.append(w)

                if w == v:
                    break

            result.append(component)

    for node in nodes:
        if node not in indices:
            visit(node)

    return result


print(strongly_connected_components(cfg))

[['Exit'], ['Body', 'Check'], ['Start']]


In [ ]:
def detect_infinite_loop(graph, entry):
    reachable = reachable_nodes(graph, entry)
    components = strongly_connected_components(graph)

    loops = []

    for component in components:
        comp = set(component) & reachable

        if not comp:
            continue

        has_cycle = (
            len(comp) > 1 or
            any(node in graph.get(node, []) for node in comp)
        )

        if has_cycle:
            loops.append(comp)

    return loops


result = detect_infinite_loop(cfg, "Start")

print("Potential Infinite Loops:", result)

Potential Infinite Loops: [{'Body', 'Check'}]


In [ ]:
def reference_program(a, b, operator):

    if operator == "add":
        return a + b

    elif operator == "sub":
        return a - b

    elif operator == "mul":
        return a * b

    elif operator == "div":
        return a / b

    else:
        return "Invalid Operator"


print(reference_program(10, 5, "add"))
print(reference_program(10, 5, "sub"))
print(reference_program(10, 5, "mul"))
print(reference_program(10, 5, "div"))

15
5
50
2.0


In [ ]:
def compiled_program(a, b, operator):

    if operator == "add":
        return a + b

    elif operator == "sub":
        return a - b

    elif operator == "mul":
        return a * b

    elif operator == "div":
        return a // b

    else:
        return "Invalid Operator"


print(compiled_program(10, 5, "add"))
print(compiled_program(10, 5, "div"))

15
2


In [ ]:
def differential_testing(test_cases):

    bugs = []

    for a, b, operator in test_cases:

        expected = reference_program(a, b, operator)
        actual = compiled_program(a, b, operator)

        if expected != actual:
            bugs.append({
                "Input": (a, b, operator),
                "Expected Output": expected,
                "Actual Output": actual
            })

    return bugs


test_cases = [
    (10, 5, "add"),
    (10, 5, "sub"),
    (10, 5, "mul"),
    (10, 5, "div"),
    (7, 2, "div"),
    (1, 4, "div")
]

bugs = differential_testing(test_cases)

print("Compiler Bug Detection Results:")

for bug in bugs:
    print(bug)

Compiler Bug Detection Results:
{'Input': (7, 2, 'div'), 'Expected Output': 3.5, 'Actual Output': 3}
{'Input': (1, 4, 'div'), 'Expected Output': 0.25, 'Actual Output': 0}


In [ ]:
machine_1 = {
    "start": "even",

    "transitions": {
        ("even", "0"): ("even", "A"),
        ("even", "1"): ("odd", "B"),
        ("odd", "0"): ("odd", "B"),
        ("odd", "1"): ("even", "A")
    }
}

print("Machine 1 Created")

Machine 1 Created


In [ ]:
machine_2 = {
    "start": "even",

    "transitions": {
        ("even", "0"): ("even", "A"),
        ("even", "1"): ("odd", "X"),
        ("odd", "0"): ("odd", "B"),
        ("odd", "1"): ("even", "A")
    }
}

print("Machine 2 Created")

Machine 2 Created


In [ ]:
def run_machine(machine, input_sequence):

    state = machine["start"]
    outputs = []

    for symbol in input_sequence:

        state, output = machine["transitions"][
            (state, symbol)
        ]

        outputs.append(output)

    return outputs


input_sequence = ["0", "1", "0", "1"]

print("Machine 1 Output:")
print(run_machine(machine_1, input_sequence))

print("Machine 2 Output:")
print(run_machine(machine_2, input_sequence))

Machine 1 Output:
['A', 'B', 'B', 'A']
Machine 2 Output:
['A', 'X', 'B', 'A']


In [ ]:
def equivalence_checker(machine_a, machine_b, alphabet):

    start = (
        machine_a["start"],
        machine_b["start"]
    )

    queue = deque([(start, [])])
    visited = {start}

    while queue:

        (state_a, state_b), sequence = queue.popleft()

        for symbol in alphabet:

            next_a, output_a = machine_a["transitions"][
                (state_a, symbol)
            ]

            next_b, output_b = machine_b["transitions"][
                (state_b, symbol)
            ]

            new_sequence = sequence + [symbol]

            if output_a != output_b:

                return {
                    "Equivalent": False,
                    "Counterexample": new_sequence,
                    "Machine A Output": run_machine(
                        machine_a, new_sequence
                    ),
                    "Machine B Output": run_machine(
                        machine_b, new_sequence
                    )
                }

            pair = (next_a, next_b)

            if pair not in visited:
                visited.add(pair)
                queue.append((pair, new_sequence))

    return {
        "Equivalent": True,
        "Counterexample": None
    }


alphabet = ["0", "1"]

result = equivalence_checker(
    machine_1,
    machine_2,
    alphabet
)

print(result)

{'Equivalent': False, 'Counterexample': ['1'], 'Machine A Output': ['B'], 'Machine B Output': ['X']}


In [ ]:
machine_3 = {
    "start": "even",
    "transitions": dict(machine_1["transitions"])
}

result = equivalence_checker(
    machine_1,
    machine_3,
    alphabet
)

print("Equivalence Result:")
print(result)

Equivalence Result:
{'Equivalent': True, 'Counterexample': None}


In [ ]:
print("INFINITE LOOP DETECTION")
print(detect_infinite_loop(cfg, "Start"))

print("\nCOMPILER BUG DETECTION")

for bug in differential_testing(test_cases):
    print(bug)

print("\nPROGRAM EQUIVALENCE CHECKING")

print(
    equivalence_checker(
        machine_1,
        machine_2,
        alphabet
    )
)

print("\nEQUIVALENT PROGRAM CHECKING")

print(
    equivalence_checker(
        machine_1,
        machine_3,
        alphabet
    )
)

INFINITE LOOP DETECTION
[{'Body', 'Check'}]

COMPILER BUG DETECTION
{'Input': (7, 2, 'div'), 'Expected Output': 3.5, 'Actual Output': 3}
{'Input': (1, 4, 'div'), 'Expected Output': 0.25, 'Actual Output': 0}

PROGRAM EQUIVALENCE CHECKING
{'Equivalent': False, 'Counterexample': ['1'], 'Machine A Output': ['B'], 'Machine B Output': ['X']}

EQUIVALENT PROGRAM CHECKING
{'Equivalent': True, 'Counterexample': None}
